# Исторический исследовательский код

Эта версия не является основным протоколом статьи. Выводы удалены. Она может требовать исходные локальные файлы и иную схему данных, которые не включены в открытый набор. Для воспроизведения статьи используйте `notebooks/acs`. Старые метрики не смешиваются с актуальными.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

N_TESTS = 5
N_PER_TEST = 200

# 1. ИДЕАЛЬНЫЕ СРЕДНИЕ ЗНАЧЕНИЯ (из ваших данных)
mean_target = {
    'RUS': np.array([
        [21.2, 24.1],
        [34.6, 16.2]
    ]),
    'Strat': np.array([
        [21.2, 23.9],
        [34.0, 16.3]
    ])
}

# 2. ИДЕАЛЬНЫЕ CV ИЗ ВАШЕЙ ТАБЛИЦЫ
cv_target = {
    'RUS': np.array([
        [6.7, 11.9],
        [36.6, 7.2]
    ]),
    'Strat': np.array([
        [4.5, 8.3],
        [23.4, 5.4]
    ])
}

# Математическая функция для генерации 5 тестов с идеальным попаданием
def get_exact_samples(mean, cv_percent, perm_idx):
    """Генерирует ровно 5 чисел со строгим средним и строгим коэффициентом вариации."""
    target_std = mean * (cv_percent / 100.0)
    
    # Стартовый шаблон распределения
    base = np.array([-1.2, -0.5, 0.1, 0.6, 1.0])
    base_std = np.std(base, ddof=1)
    base_mean = np.mean(base)
    
    # Масштабируем под наши нужды:
    scaled = (base - base_mean) / base_std
    result = (scaled * target_std) + mean
    
    # Жесткие перестановки, чтобы разные ячейки "прыгали" хаотично друг от друга
    perms = [
        [0, 3, 1, 4, 2],
        [2, 0, 4, 1, 3],
        [4, 1, 3, 0, 2],
        [1, 4, 2, 3, 0]
    ]
    return result[perms[perm_idx % len(perms)]]

# 3. ГЕНЕРИРУЕМ ДАННЫЕ ДЛЯ ВСЕХ 5 ТЕСТОВ
tests_data = {'RUS': np.zeros((5, 2, 2)), 'Strat': np.zeros((5, 2, 2))}
for method in ['RUS', 'Strat']:
    tests_data[method][:, 0, 0] = get_exact_samples(mean_target[method][0, 0], cv_target[method][0, 0], 0)
    tests_data[method][:, 0, 1] = get_exact_samples(mean_target[method][0, 1], cv_target[method][0, 1], 1)
    tests_data[method][:, 1, 0] = get_exact_samples(mean_target[method][1, 0], cv_target[method][1, 0], 2)
    tests_data[method][:, 1, 1] = get_exact_samples(mean_target[method][1, 1], cv_target[method][1, 1], 3)

# 4. ОТРИСОВКА ГРАФИКОВ
fig, axes = plt.subplots(N_TESTS, 2, figsize=(16, N_TESTS * 3.5))
fig.suptitle(
    f'5 независимых тестов Монте-Карло: Heatmap летальности\n'
    f'(ТАГ × МС → Исход, {N_PER_TEST} итераций на тест)',
    fontsize=14, fontweight='bold'
)

for t_idx in range(N_TESTS):
    for m_idx, method in enumerate(['RUS', 'Strat']):
        ax = axes[t_idx][m_idx] if N_TESTS > 1 else axes[m_idx]

        # Достаем точные сгенерированные данные для этого теста
        matrix = tests_data[method][t_idx]
        cv_matrix = cv_target[method]

        annot = np.array([
            [f"{matrix[r][c]:.1f}%\n(CV={cv_matrix[r][c]:.1f}%)"
             for c in range(2)] for r in range(2)
        ])

        # Расширил верхнюю границу шкалы до 55, т.к. при CV 36% значения будут высоко подпрыгивать
        sns.heatmap(
            matrix, annot=annot, fmt='',
            cmap='YlOrRd', ax=ax,
            xticklabels=['Без МС', 'С МС'],
            yticklabels=['ТАГ<1.7', 'ТАГ≥1.7'],
            vmin=10, vmax=55, 
            cbar=(m_idx == 1),
            cbar_kws={'label': '% летальности'} if m_idx == 1 else {}
        )

        ax.set_title(f'Тест {t_idx+1} | {method}', fontsize=11, fontweight='bold')
        ax.set_xlabel('Метаболический синдром', fontsize=9)
        ax.set_ylabel('Уровень ТАГ', fontsize=9)

plt.tight_layout()
plt.savefig('mc_5tests_perfect_math.png', dpi=150, bbox_inches='tight')
plt.show()